# AI Enterprise Workflow Capstone: Multi-Model Evaluation & Baseline Comparison
**Project Title:** AI Enterprise Revenue Forecasting & Production Monitoring Microservice

This notebook satisfies the rubric requirements for:
- **Multiple model comparison** (Baseline Dummy, Ridge Regression, Random Forest, Gradient Boosting)
- **Visualization comparing candidate model to baseline**
- Cross-validation error metrics (RMSE, MAE, R²)

In [1]:
from data_ingestion import fetch_data, convert_to_ts
from model import compare_models, FEATURES, TARGET, build_model_pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_raw = fetch_data()
ts_all = convert_to_ts(df_raw, country='all')

comp_results = compare_models(ts_all)
pd.DataFrame(comp_results).T

### Visualizing Multi-Model Metrics vs Baseline

In [2]:
models = list(comp_results.keys())
rmse_vals = [comp_results[m]['RMSE'] for m in models]
mae_vals = [comp_results[m]['MAE'] for m in models]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))
width = 0.35
ax.bar(x - width/2, rmse_vals, width, label='RMSE', color='#e377c2')
ax.bar(x + width/2, mae_vals, width, label='MAE', color='#17becf')
ax.set_ylabel('Error Value (Lower is Better)')
ax.set_title('Multiple Model Comparison vs Baseline (Cross-Validation)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
plt.tight_layout()
plt.show()

### Visualizing Hold-Out Predictions: Candidate Model vs Baseline Model

In [3]:
clean_df = ts_all.dropna(subset=FEATURES + [TARGET]).iloc[30:-30].copy()
X = clean_df[FEATURES]
y = clean_df[TARGET]
split_idx = int(len(clean_df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = clean_df['date'].iloc[split_idx:]

baseline_pred = np.full(shape=len(y_test), fill_value=y_train.mean())
rf_pipe = build_model_pipeline()
rf_pipe.fit(X_train, y_train)
rf_preds = rf_pipe.predict(X_test)

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(dates_test, y_test, label='Actual 30-Day Revenue', color='#333333', linewidth=2.0)
ax.plot(dates_test, baseline_pred, label='Baseline Model (Mean)', color='#d62728', linestyle='--', linewidth=2.0)
ax.plot(dates_test, rf_preds, label='Production Model (Random Forest)', color='#2ca02c', linewidth=2.2)
ax.set_title('Hold-Out Comparison: Production Model vs Baseline')
ax.set_xlabel('Target Date')
ax.set_ylabel('30-Day Forward Revenue ($)')
ax.legend()
plt.tight_layout()
plt.show()